## 1. Persistence Baseline

In [1]:
import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive

    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/drive/MyDrive/dissertation')
else:
    PROJECT_ROOT = Path('../..').resolve()

sys.path.insert(0, str(PROJECT_ROOT))

print(f"IN_COLAB={IN_COLAB}  REPO_ROOT={PROJECT_ROOT}")

IN_COLAB=False  REPO_ROOT=/Users/ucabtt4/Projects/Dissertation/dissertation


In [12]:
import json
import numpy as np

from src.dataloader import load_arrays, load_cgm_stats, load_participant_ids
from src.metrics import evaluate, format_results_table, evaluate_per_participant
from src.models.classical_baseline import PersistencePredictor, HORIZONS

MODEL_NAME = 'Persistence'
EXPERIMENT_DIR = PROJECT_ROOT / 'experiments' / '1_baseline'
RESULTS_DIR = EXPERIMENT_DIR / 'results' / 'persistence'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# store results in persistence
RESULTS_PATH = RESULTS_DIR / 'persistence_results.json'
PARTICIPANT_SUMMARY_PATH = RESULTS_DIR / 'persistence_participant_stability_summary.json'

In [6]:
stats = load_cgm_stats()
X_train, y_train = load_arrays('train')
X_val, y_val = load_arrays('val')
X_test, y_test = load_arrays('test')

test_participant_ids = load_participant_ids('test')
PARTICIPANT_IDS = sorted(set(test_participant_ids))

print('train set shape: ', X_train.shape, '\nvaluation set shape: ', X_val.shape, '\ntest set shape: ', X_test.shape)
print('\nnumber of test participants: ', len(PARTICIPANT_IDS))
print('Corresponding participants ID: \n', PARTICIPANT_IDS)

train set shape:  (83532, 24) 
valuation set shape:  (16739, 24) 
test set shape:  (16842, 24)

number of test participants:  6
Corresponding participants ID: 
 [np.str_('CGMacros-008'), np.str_('CGMacros-014'), np.str_('CGMacros-019'), np.str_('CGMacros-020'), np.str_('CGMacros-034'), np.str_('CGMacros-039')]


In [7]:
model = PersistencePredictor().fit(X_train, y_train)  # nothing to fit -- zero parameters

In [ ]:
# Evaluate Linear Regression
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

val_pred = model.predict(X_val)
test_pred = model.predict(X_test)

val_results = evaluate(
    val_pred, y_val,
    stats['cgm_mean'], stats['cgm_std']
)

test_results = evaluate(
    test_pred, y_test,
    stats['cgm_mean'], stats['cgm_std']
)

test_results_per_participant = evaluate_per_participant(
    test_pred, y_test, test_participant_ids,
    stats['cgm_mean'], stats['cgm_std']
)


In [17]:
# Save complete results
result = {
    'model': MODEL_NAME,
    'seed': None,  # Linear Regression is deterministic
    'val_results': val_results,
    'test_results': test_results,
    'test_results_per_participant': test_results_per_participant,
}

with open(RESULTS_PATH, 'w') as f:
    json.dump(result, f, indent=2)

# Simple test-set output
print(f"\nTest results — {MODEL_NAME}\n")

for h in HORIZONS:
    r = test_results[h]
    zone_a = r['ceg']['A']
    zone_ab = r['ceg']['A'] + r['ceg']['B']

    print(
        f"{h:>3} min | "
        f"RMSE {r['rmse']:.2f} | "
        f"MAE {r['mae']:.2f} | "
        f"gRMSE {r['grmse']:.2f} | "
        f"Zone A {zone_a:.2f}% | "
        f"A+B {zone_ab:.2f}%"
    )

print(f"\n{'-' * 80}")
print(f"saved to {RESULTS_PATH}")


Test results — Persistence

 15 min | RMSE 12.78 | MAE 8.33 | gRMSE 14.10 | Zone A 95.97% | A+B 99.86%
 30 min | RMSE 20.49 | MAE 13.23 | gRMSE 23.20 | Zone A 88.87% | A+B 99.65%
 60 min | RMSE 30.43 | MAE 19.47 | gRMSE 35.32 | Zone A 79.38% | A+B 98.98%
 90 min | RMSE 37.39 | MAE 24.03 | gRMSE 43.73 | Zone A 72.82% | A+B 98.13%
120 min | RMSE 42.09 | MAE 27.34 | gRMSE 49.28 | Zone A 68.32% | A+B 97.22%

--------------------------------------------------------------------------------
saved to /Users/ucabtt4/Projects/Dissertation/dissertation/experiments/1_baseline/results/persistence/persistence_results.json


In [18]:
def _extract(result, metric):
    if metric == 'zone_a':
        return result['ceg']['A']
    if metric == 'zone_ab':
        return result['ceg']['A'] + result['ceg']['B']
    return result[metric]


METRICS = ['rmse', 'mae', 'grmse', 'zone_a', 'zone_ab']

participant_summary = {}

for h in HORIZONS:
    participant_summary[h] = {}

    for m in METRICS:

        # Main mean: pooled across all test samples
        mean = _extract(test_results[h], m)

        # SD1: no seed variability for deterministic Linear Regression
        sd1 = 0.0

        # SD2: variability across participants
        participant_values = np.array([
            _extract(test_results_per_participant[pid][h], m)
            for pid in PARTICIPANT_IDS
        ])

        sd2 = participant_values.std(ddof=1)

        participant_summary[h][m] = {
            'mean': float(mean),
            'sd1_across_seeds': float(sd1),
            'sd2_across_participants': float(sd2),
        }


print(
    f"=== {MODEL_NAME} test-set: mean +/- sd1 (sd2) | "
    f"sd1=0 (deterministic), sd2 across {len(PARTICIPANT_IDS)} participants ==="
)

for h in HORIZONS:
    s = participant_summary[h]

    print(
        f"{h:>3} min | "
        f"RMSE {s['rmse']['mean']:.2f} +/- {s['rmse']['sd1_across_seeds']:.2f} "
        f"({s['rmse']['sd2_across_participants']:.2f}) | "
        f"MAE {s['mae']['mean']:.2f} +/- {s['mae']['sd1_across_seeds']:.2f} "
        f"({s['mae']['sd2_across_participants']:.2f}) | "
        f"gRMSE {s['grmse']['mean']:.2f} +/- {s['grmse']['sd1_across_seeds']:.2f} "
        f"({s['grmse']['sd2_across_participants']:.2f}) | "
        f"Zone A {s['zone_a']['mean']:.2f} +/- {s['zone_a']['sd1_across_seeds']:.2f} "
        f"({s['zone_a']['sd2_across_participants']:.2f})% | "
        f"A+B {s['zone_ab']['mean']:.2f} +/- {s['zone_ab']['sd1_across_seeds']:.2f} "
        f"({s['zone_ab']['sd2_across_participants']:.2f})%"
    )

=== Persistence test-set: mean +/- sd1 (sd2) | sd1=0 (deterministic), sd2 across 6 participants ===
 15 min | RMSE 12.78 +/- 0.00 (2.29) | MAE 8.33 +/- 0.00 (1.72) | gRMSE 14.10 +/- 0.00 (3.45) | Zone A 95.97 +/- 0.00 (1.98)% | A+B 99.86 +/- 0.00 (0.19)%
 30 min | RMSE 20.49 +/- 0.00 (4.73) | MAE 13.23 +/- 0.00 (3.30) | gRMSE 23.20 +/- 0.00 (6.94) | Zone A 88.87 +/- 0.00 (2.99)% | A+B 99.65 +/- 0.00 (0.31)%
 60 min | RMSE 30.43 +/- 0.00 (10.23) | MAE 19.47 +/- 0.00 (6.86) | gRMSE 35.32 +/- 0.00 (14.06) | Zone A 79.38 +/- 0.00 (5.41)% | A+B 98.98 +/- 0.00 (0.87)%
 90 min | RMSE 37.39 +/- 0.00 (14.43) | MAE 24.03 +/- 0.00 (9.71) | gRMSE 43.73 +/- 0.00 (19.51) | Zone A 72.82 +/- 0.00 (7.95)% | A+B 98.13 +/- 0.00 (1.99)%
120 min | RMSE 42.09 +/- 0.00 (17.54) | MAE 27.34 +/- 0.00 (11.99) | gRMSE 49.28 +/- 0.00 (23.42) | Zone A 68.32 +/- 0.00 (10.20)% | A+B 97.22 +/- 0.00 (3.35)%


In [19]:
with open(PARTICIPANT_SUMMARY_PATH, 'w') as f:
    json.dump({
        'model': MODEL_NAME,
        'participant_ids': PARTICIPANT_IDS,
        'per_horizon_summary': participant_summary,
    }, f, indent=2)

print(f"\nsaved to {PARTICIPANT_SUMMARY_PATH}")


saved to /Users/ucabtt4/Projects/Dissertation/dissertation/experiments/1_baseline/results/persistence/persistence_participant_stability_summary.json
